# 03 · ExcelFormer Baseline (Home Credit)

ExcelFormer — a transformer architecture designed specifically for tabular data — applied to the Home Credit Default Risk task. Used as a non-FT-Transformer transformer baseline alongside SAINT (`02_saint_baseline.ipynb`).

## 1. Setup

In [ ]:
# %pip install rtdl_revisiting_models -q
%pip install delu -q
%pip install optuna -q

In [ ]:
import math
import random
import warnings
import json
import os
import itertools
import typing
from collections import OrderedDict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Literal, Optional, Tuple, cast

import numpy as np
import pandas as pd
import scipy.special

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch import Tensor
from torch.nn.parameter import Parameter
from torch.utils.data import Dataset, DataLoader, TensorDataset

import sklearn.datasets
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing
import sklearn.tree as sklearn_tree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, QuantileTransformer
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression

import optuna
import delu
from tqdm.std import tqdm
from tqdm import tqdm as _tqdm  # alias for cells that use it directly

warnings.filterwarnings("ignore")
pd.options.display.max_rows = 200
pd.options.display.max_columns = 200

RANDOM_SEED = 42


def seed_everything(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# --- Optional: Google Colab Drive mount ---
# Uncomment the three lines below if you're running on Colab and want to read
# data from / write artifacts to your Drive. Skip on local, server, or Kaggle
# runs. The next cell auto-routes through DRIVE_ROOT when defined.
#
# from google.colab import drive
# drive.mount("/content/drive")
# DRIVE_ROOT = "/content/drive/MyDrive/ft-transformer-credit-risk-study"

# When running locally, repo root is one directory up (notebook is in
# notebooks/<dataset>/). On Colab with the cell above uncommented, DRIVE_ROOT
# takes precedence.
_BASE = globals().get("DRIVE_ROOT", "..")
DATA_PATH      = f"{_BASE}/data/preprocessed_data_home_credit.parquet.gzip"
ARTIFACTS_DIR  = Path(f"{_BASE}/artifacts/home_credit")
RESULTS_DIR    = ARTIFACTS_DIR / "results"
MODELS_DIR     = ARTIFACTS_DIR / "models"
FIGURES_DIR    = ARTIFACTS_DIR / "figures"
for _d in (ARTIFACTS_DIR, RESULTS_DIR, MODELS_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"DATA_PATH      = {DATA_PATH}")
print(f"ARTIFACTS_DIR  = {ARTIFACTS_DIR}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# DEV_MODE — smoke-test switch.
#
# When False (the default), every constant resolves to the exact value used in
# the original experiment. The DEV_MODE branches below are dead code in that
# path, so the run is behaviourally identical to the source notebook.
#
# When True, the notebook runs a tiny fast version: a subsample of the data,
# a couple of epochs, a single Optuna trial. Use this on Colab to validate the
# full pipeline end-to-end before launching a real run.
# ──────────────────────────────────────────────────────────────────────────────
DEV_MODE = False
if DEV_MODE:
    print("DEV_MODE is ON — smoke-test run with reduced data/epochs/trials.")

## 2. Data Loading

In [ ]:
# Load preprocessed Home Credit data + drop the leakage / low-signal columns
# identified during initial EDA.
df = pd.read_parquet(DATA_PATH)

DROP_COLUMNS_HC = [
    "FLAG_PHONE", "OCCUPATION_TYPE", "SELLERPLACE_AREA", "AMT_CREDIT_LIMIT_ACTUAL",
    "AMT_DRAWINGS_ATM_CURRENT", "CODE_REJECT_REASON_HC", "AMT_PAYMENT_TOTAL_CURRENT",
    "CHANNEL_TYPE_AP+ (Cash loan)", "NUM_INSTALMENT_NUMBER_x", "CNT_INSTALMENT_MATURE_CUM",
    "CREDIT_TYPE_Car loan", "WEEKDAY_APPR_PROCESS_START_MONDAY",
    "CHANNEL_TYPE_Credit and cash offices", "NAME_YIELD_GROUP_middle",
    "AMT_DRAWINGS_CURRENT", "SK_ID_PREV_y", "CHANNEL_TYPE_Channel of corporate sales",
    "SK_DPD_y", "WEEKDAY_APPR_PROCESS_START_WEDNESDAY", "MONTHS_MAX",
    "NAME_GOODS_CATEGORY_Clothing and Accessories", "CODE_REJECT_REASON_SCO",
    "OBS_30_CNT_SOCIAL_CIRCLE", "AMT_REQ_CREDIT_BUREAU_YEAR",
    "PRODUCT_COMBINATION_Cash Street: middle",
    "NAME_GOODS_CATEGORY_Photo / Cinema Equipment",
    "PRODUCT_COMBINATION_Cash X-Sell: middle", "AMT_APPLICATION", "DAYS_FIRST_DUE",
    "FLAG_DOCUMENT_16", "AMT_ANNUITY", "NAME_SELLER_INDUSTRY_Connectivity",
    "PRODUCT_COMBINATION_POS industry without interest", "NONLIVINGAPARTMENTS_AVG",
    "NAME_SELLER_INDUSTRY_Industry", "DAYS_CREDIT_UPDATE", "RATE_DOWN_PAYMENT",
    "ELEVATORS_AVG", "STATUS_X", "ENTRANCES_AVG",
    "WEEKDAY_APPR_PROCESS_START_SATURDAY", "MONTHS_BALANCE_y",
    "AMT_CREDIT_SUM_OVERDUE", "NAME_PORTFOLIO_POS", "SK_DPD_x", "FLAG_DOCUMENT_13",
    "NAME_TYPE_SUITE_Unaccompanied", "WEEKDAY_APPR_PROCESS_START_FRIDAY",
    "FLAG_DOCUMENT_6", "CHANNEL_TYPE_Stone", "SK_DPD_DEF_y",
    "NAME_PRODUCT_TYPE_x-sell", "NONLIVINGAREA_AVG", "FLAG_DOCUMENT_18",
    "nb_app", "NAME_HOUSING_TYPE", "NAME_PORTFOLIO_Cash",
    "WEEKDAY_APPR_PROCESS_START_SUNDAY", "WEEKDAY_APPR_PROCESS_START_THURSDAY",
    "NFLAG_INSURED_ON_APPROVAL", "MONTHS_COUNT", "YEARS_BUILD_MEDI",
    "DAYS_FIRST_DRAWING", "CODE_REJECT_REASON_LIMIT", "NAME_YIELD_GROUP_XNA",
    "NAME_GOODS_CATEGORY_Medical Supplies", "CREDIT_TYPE_Consumer credit",
    "PRODUCT_COMBINATION_POS household with interest",
    "PRODUCT_COMBINATION_POS mobile with interest", "AMT_DRAWINGS_POS_CURRENT",
    "NAME_CONTRACT_TYPE_Revolving loans", "FLOORSMAX_AVG", "WALLSMATERIAL_MODE",
    "CHANNEL_TYPE_Contact center", "CREDIT_TYPE_Credit card",
    "CODE_REJECT_REASON_SCOFR", "NAME_SELLER_INDUSTRY_XNA", "BASEMENTAREA_AVG",
    "LANDAREA_MEDI", "NAME_SELLER_INDUSTRY_Consumer electronics", "NAME_TYPE_SUITE",
    "FLAG_LAST_APPL_PER_CONTRACT_Y", "CNT_CHILDREN",
    "PRODUCT_COMBINATION_Cash Street: high", "NAME_GOODS_CATEGORY_Sport and Leisure",
    "NAME_TYPE_SUITE_Children", "NAME_SELLER_INDUSTRY_Construction",
    "NAME_GOODS_CATEGORY_Tourism", "NAME_CASH_LOAN_PURPOSE_Repairs",
    "FLAG_CONT_MOBILE", "WEEKDAY_APPR_PROCESS_START_TUESDAY",
    "AMT_REQ_CREDIT_BUREAU_DAY", "NAME_GOODS_CATEGORY_Mobile",
    "NAME_TYPE_SUITE_Spouse, partner", "REG_REGION_NOT_LIVE_REGION",
    "CREDIT_TYPE_Another type of loan", "COMMONAREA_MEDI", "FONDKAPREMONT_MODE",
    "NAME_GOODS_CATEGORY_Construction Materials", "NAME_PORTFOLIO_XNA",
    "NAME_TYPE_SUITE_Family", "NAME_PAYMENT_TYPE_Non-cash from your account",
    "NAME_CASH_LOAN_PURPOSE_Other", "CNT_FAM_MEMBERS", "CREDIT_ACTIVE_Sold",
    "PRODUCT_COMBINATION_Card Street", "PRODUCT_COMBINATION_Cash",
    "NAME_TYPE_SUITE_Other_B", "NAME_CONTRACT_TYPE_Cash loans", "FLOORSMIN_AVG",
    "CHANNEL_TYPE_Country-wide", "NUM_INSTALMENT_VERSION",
    "NAME_CONTRACT_STATUS_Canceled", "AMT_REQ_CREDIT_BUREAU_MON",
    "NAME_TYPE_SUITE_Other_A", "NAME_SELLER_INDUSTRY_MLM partners",
    "NAME_GOODS_CATEGORY_Gardening", "REG_CITY_NOT_WORK_CITY",
    "CNT_CREDIT_PROLONG", "NUM_INSTALMENT_NUMBER",
    "NAME_GOODS_CATEGORY_Auto Accessories", "STATUS_C", "NAME_GOODS_CATEGORY_Jewelry",
    "AMT_REQ_CREDIT_BUREAU_WEEK", "CHANNEL_TYPE_Car dealer", "HOUSETYPE_MODE",
    "FLAG_MOBIL", "FLAG_EMAIL", "FLAG_EMP_PHONE",
    "LIVE_REGION_NOT_WORK_REGION", "LIVE_CITY_NOT_WORK_CITY",
    "NAME_GOODS_CATEGORY_Fitness", "NAME_GOODS_CATEGORY_Education",
    "NAME_GOODS_CATEGORY_Direct Sales", "NAME_GOODS_CATEGORY_Consumer Electronics",
    "NAME_GOODS_CATEGORY_Computers", "NAME_GOODS_CATEGORY_Audio/Video",
    "NAME_GOODS_CATEGORY_Additional Service", "NAME_GOODS_CATEGORY_Animals",
    "NAME_TYPE_SUITE_Group of people", "NAME_CLIENT_TYPE_Refreshed",
    "NAME_CLIENT_TYPE_XNA", "AMT_DRAWINGS_OTHER_CURRENT",
    "CREDIT_TYPE_Unknown type of loan", "CODE_REJECT_REASON_SYSTEM",
    "CODE_REJECT_REASON_XNA", "CODE_REJECT_REASON_VERIF",
    "CREDIT_TYPE_Real estate loan", "NUNIQUE_STATUS_x", "NUNIQUE_STATUS_y",
    "NUNIQUE_STATUS2_y", "CNT_DRAWINGS_OTHER_CURRENT",
    "REG_REGION_NOT_WORK_REGION", "NAME_GOODS_CATEGORY_House Construction",
    "NAME_GOODS_CATEGORY_Homewares", "NAME_CASH_LOAN_PURPOSE_Buying a new car",
    "NAME_CASH_LOAN_PURPOSE_Buying a used car", "NAME_CASH_LOAN_PURPOSE_Car repairs",
    "NAME_CASH_LOAN_PURPOSE_Education",
    "NAME_CASH_LOAN_PURPOSE_Business development",
    "NAME_CASH_LOAN_PURPOSE_Buying a garage", "FLAG_LAST_APPL_PER_CONTRACT_N",
    "NAME_CASH_LOAN_PURPOSE_Building a house or an annex",
    "NFLAG_LAST_APPL_IN_DAY", "CHANNEL_TYPE_Regional / Local",
    "NAME_SELLER_INDUSTRY_Auto technology", "FLAG_DOCUMENT_17",
    "FLAG_DOCUMENT_19", "FLAG_DOCUMENT_20", "FLAG_DOCUMENT_21",
    "AMT_REQ_CREDIT_BUREAU_HOUR", "NAME_GOODS_CATEGORY_Other",
    "NAME_GOODS_CATEGORY_Office Appliances", "NAME_GOODS_CATEGORY_Insurance",
    "NAME_GOODS_CATEGORY_Medicine", "NAME_GOODS_CATEGORY_Weapon",
    "NAME_GOODS_CATEGORY_Vehicles", "NAME_CASH_LOAN_PURPOSE_Buying a holiday home / land",
    "NAME_CASH_LOAN_PURPOSE_Buying a home", "NAME_CASH_LOAN_PURPOSE_Hobby",
    "NAME_CASH_LOAN_PURPOSE_Gasification / water supply",
    "NAME_CASH_LOAN_PURPOSE_Furniture", "NAME_CASH_LOAN_PURPOSE_Everyday expenses",
    "NAME_CASH_LOAN_PURPOSE_Money for a third person",
    "NAME_CASH_LOAN_PURPOSE_Payments on other loans",
    "NAME_CASH_LOAN_PURPOSE_Medicine", "NAME_CASH_LOAN_PURPOSE_Journey",
    "NAME_CASH_LOAN_PURPOSE_Refusal to name the goal",
    "NAME_CASH_LOAN_PURPOSE_Purchase of electronic equipment",
    "NAME_CASH_LOAN_PURPOSE_Wedding / gift / holiday",
    "NAME_CASH_LOAN_PURPOSE_Urgent needs",
    "NAME_CONTRACT_STATUS_Unused offer",
    "NAME_PAYMENT_TYPE_Cashless from the account of the employer",
    "NAME_CONTRACT_TYPE_XNA",
    "PRODUCT_COMBINATION_POS household without interest",
    "FLAG_DOCUMENT_5", "FLAG_DOCUMENT_4", "FLAG_DOCUMENT_2",
    "EMERGENCYSTATE_MODE",
    "PRODUCT_COMBINATION_POS others without interest",
    "PRODUCT_COMBINATION_POS other with interest",
    "PRODUCT_COMBINATION_POS mobile without interest", "CREDIT_DAY_OVERDUE",
    "NAME_SELLER_INDUSTRY_Tourism", "NAME_SELLER_INDUSTRY_Jewelry",
    "PRODUCT_COMBINATION_Card X-Sell", "FLAG_DOCUMENT_15",
    "FLAG_DOCUMENT_7", "FLAG_DOCUMENT_8", "FLAG_DOCUMENT_9",
    "FLAG_DOCUMENT_10", "CREDIT_CURRENCY_currency 4",
    "CREDIT_CURRENCY_currency 3", "CREDIT_CURRENCY_currency 2",
    "CREDIT_ACTIVE_Bad debt", "FLAG_DOCUMENT_11", "FLAG_DOCUMENT_12",
    "FLAG_DOCUMENT_14", "CREDIT_TYPE_Cash loan (non-earmarked)",
    "CREDIT_TYPE_Loan for the purchase of equipment",
    "CREDIT_TYPE_Loan for purchase of shares (margin lending)",
    "CREDIT_TYPE_Loan for business development",
    "CREDIT_TYPE_Interbank credit",
    "CREDIT_TYPE_Loan for working capital replenishment",
    "CREDIT_TYPE_Mobile operator loan", "FLAG_OWN_REALTY", "FLAG_OWN_CAR",
]
df = df.drop(columns=DROP_COLUMNS_HC)

NUM_COLS_HC = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT_x", "AMT_ANNUITY_x", "AMT_GOODS_PRICE_x",
    "REGION_POPULATION_RELATIVE", "DAYS_BIRTH", "DAYS_EMPLOYED",
    "DAYS_REGISTRATION", "DAYS_ID_PUBLISH", "OWN_CAR_AGE",
    "REGION_RATING_CLIENT", "REGION_RATING_CLIENT_W_CITY",
    "HOUR_APPR_PROCESS_START_x", "REG_CITY_NOT_LIVE_CITY",
    "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3",
    "APARTMENTS_AVG", "YEARS_BEGINEXPLUATATION_AVG", "YEARS_BUILD_AVG",
    "COMMONAREA_AVG", "LANDAREA_AVG", "LIVINGAPARTMENTS_AVG", "LIVINGAREA_AVG",
    "APARTMENTS_MODE", "BASEMENTAREA_MODE", "YEARS_BEGINEXPLUATATION_MODE",
    "YEARS_BUILD_MODE", "COMMONAREA_MODE", "ELEVATORS_MODE", "ENTRANCES_MODE",
    "FLOORSMAX_MODE", "FLOORSMIN_MODE", "LANDAREA_MODE", "LIVINGAPARTMENTS_MODE",
    "LIVINGAREA_MODE", "NONLIVINGAPARTMENTS_MODE", "NONLIVINGAREA_MODE",
    "APARTMENTS_MEDI", "BASEMENTAREA_MEDI", "YEARS_BEGINEXPLUATATION_MEDI",
    "ELEVATORS_MEDI", "ENTRANCES_MEDI", "FLOORSMAX_MEDI", "FLOORSMIN_MEDI",
    "LIVINGAPARTMENTS_MEDI", "LIVINGAREA_MEDI", "NONLIVINGAPARTMENTS_MEDI",
    "NONLIVINGAREA_MEDI", "TOTALAREA_MODE", "DEF_30_CNT_SOCIAL_CIRCLE",
    "OBS_60_CNT_SOCIAL_CIRCLE", "DEF_60_CNT_SOCIAL_CIRCLE",
    "DAYS_LAST_PHONE_CHANGE", "AMT_REQ_CREDIT_BUREAU_QRT", "AMT_ANNUITY_y",
    "AMT_CREDIT_y", "AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE_y",
    "HOUR_APPR_PROCESS_START_y", "RATE_INTEREST_PRIMARY",
    "RATE_INTEREST_PRIVILEGED", "DAYS_DECISION", "CNT_PAYMENT",
    "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION",
    "NAME_CONTRACT_TYPE_Consumer loans", "NAME_CASH_LOAN_PURPOSE_XAP",
    "NAME_CASH_LOAN_PURPOSE_XNA", "NAME_CONTRACT_STATUS_Approved",
    "NAME_CONTRACT_STATUS_Refused", "NAME_PAYMENT_TYPE_Cash through the bank",
    "NAME_PAYMENT_TYPE_XNA", "CODE_REJECT_REASON_CLIENT",
    "CODE_REJECT_REASON_XAP", "NAME_CLIENT_TYPE_New",
    "NAME_CLIENT_TYPE_Repeater", "NAME_GOODS_CATEGORY_Furniture",
    "NAME_GOODS_CATEGORY_XNA", "NAME_PORTFOLIO_Cards", "NAME_PORTFOLIO_Cars",
    "NAME_PRODUCT_TYPE_XNA", "NAME_PRODUCT_TYPE_walk-in",
    "NAME_SELLER_INDUSTRY_Clothing", "NAME_SELLER_INDUSTRY_Furniture",
    "NAME_YIELD_GROUP_high", "NAME_YIELD_GROUP_low_action",
    "NAME_YIELD_GROUP_low_normal", "PRODUCT_COMBINATION_Cash Street: low",
    "PRODUCT_COMBINATION_Cash X-Sell: high", "PRODUCT_COMBINATION_Cash X-Sell: low",
    "PRODUCT_COMBINATION_POS industry with interest", "DAYS_CREDIT",
    "DAYS_CREDIT_ENDDATE", "DAYS_ENDDATE_FACT", "AMT_CREDIT_MAX_OVERDUE",
    "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_LIMIT",
    "STATUS_0", "STATUS_1", "STATUS_2", "STATUS_3", "STATUS_4", "STATUS_5",
    "MONTHS_MIN", "CREDIT_ACTIVE_Active", "CREDIT_ACTIVE_Closed",
    "CREDIT_CURRENCY_currency 1", "CREDIT_TYPE_Microloan", "CREDIT_TYPE_Mortgage",
    "buro_count", "MONTHS_BALANCE_x", "CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE",
    "SK_DPD_DEF_x", "NUNIQUE_STATUS2_x", "AMT_BALANCE",
    "AMT_INST_MIN_REGULARITY", "AMT_PAYMENT_CURRENT",
    "AMT_RECEIVABLE_PRINCIPAL", "AMT_RECIVABLE", "AMT_TOTAL_RECEIVABLE",
    "CNT_DRAWINGS_ATM_CURRENT", "CNT_DRAWINGS_CURRENT",
    "CNT_DRAWINGS_POS_CURRENT", "NUM_INSTALMENT_VERSION_x",
    "DAYS_INSTALMENT_x", "DAYS_ENTRY_PAYMENT_x", "AMT_INSTALMENT_x",
    "AMT_PAYMENT_x", "SK_ID_PREV_x", "NUM_INSTALMENT_VERSION_y",
    "NUM_INSTALMENT_NUMBER_y", "DAYS_INSTALMENT_y", "DAYS_ENTRY_PAYMENT_y",
    "AMT_INSTALMENT_y", "AMT_PAYMENT_y", "DAYS_INSTALMENT",
    "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT",
]
CAT_COLS_HC = [
    "NAME_CONTRACT_TYPE", "CODE_GENDER", "NAME_INCOME_TYPE",
    "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
    "WEEKDAY_APPR_PROCESS_START", "ORGANIZATION_TYPE", "EMERGENCYSTATE_MODE",
]
num_cols = NUM_COLS_HC
cat_cols = CAT_COLS_HC

## 3. Preprocessing Pipeline

In [ ]:
def apply_missing_value_filter(X_train, X_valid, X_test, threshold=0.50):
    """Drop columns whose missing rate in X_train exceeds `threshold`."""
    missing_pct = X_train.isnull().mean()
    cols_to_drop = missing_pct[missing_pct > threshold].index.tolist()
    print("--- 2. Missing Value Column Filter ---")
    print(f"Missing percentage threshold: {threshold:.1%}")
    print(f"Columns dropped ({len(cols_to_drop)}): {cols_to_drop if cols_to_drop else 'None'}")

    X_train_clean = X_train.drop(columns=cols_to_drop, errors="ignore")
    X_valid_clean = X_valid.drop(columns=cols_to_drop, errors="ignore")
    X_test_clean  = X_test.drop(columns=cols_to_drop, errors="ignore")

    all_cols = X_train_clean.columns.tolist()
    num_cols_clean = X_train_clean.select_dtypes(include=np.number).columns.tolist()
    cat_cols_clean = [c for c in all_cols if c not in num_cols_clean]
    print(f"Features remaining: {X_train_clean.shape[1]}")
    print("-" * 50)
    return X_train_clean, X_valid_clean, X_test_clean, num_cols_clean, cat_cols_clean


def apply_correlation_drop(X_train, y_train, X_valid, X_test, threshold=0.9):
    """Drop one feature from each numeric pair whose |correlation| > threshold,
    keeping the one with higher absolute correlation to the target."""
    print("--- 3. Correlation Drop (numeric only) ---")
    all_cols = X_train.columns.tolist()
    num_cols_start = X_train.select_dtypes(include=np.number).columns.tolist()
    cat_cols_start = [c for c in all_cols if c not in num_cols_start]
    X_train_num = X_train[num_cols_start]

    print(f"Numeric features before drop: {len(num_cols_start)}. Threshold: {threshold}")
    corr_matrix = X_train_num.corr().abs()
    target_corr = X_train_num.apply(lambda x: x.corr(y_train)).abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    to_drop_num = set()
    for col in upper.columns:
        for row in upper.index:
            if upper.loc[row, col] > threshold:
                a, b = row, col
                if a in to_drop_num or b in to_drop_num:
                    continue
                if target_corr.get(a, 0) < target_corr.get(b, 0):
                    to_drop_num.add(a)
                else:
                    to_drop_num.add(b)

    features_to_drop = list(to_drop_num)
    X_train_clean = X_train.drop(columns=features_to_drop, errors="ignore")
    X_valid_clean = X_valid.drop(columns=features_to_drop, errors="ignore")
    X_test_clean  = X_test.drop(columns=features_to_drop, errors="ignore")
    print(f"Numeric features dropped: {len(features_to_drop)}")
    print("-" * 50)

    num_cols_clean = [c for c in num_cols_start if c not in features_to_drop]
    cat_cols_clean = cat_cols_start
    return X_train_clean, X_valid_clean, X_test_clean, num_cols_clean, cat_cols_clean


def preprocess_data_pipeline(df, target="TARGET", corr_threshold=0.9, missing_threshold=0.50):
    """Full HC preprocessing pipeline: split, filter missing, drop correlated,
    impute, label-encode, scale. Returns processed splits and helper objects."""
    print("--- 1. Splitting Data (64/16/20) ---")
    X = df.drop(columns=[target])
    y = df[target]
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=42,
    )
    X_train, X_valid, y_train, y_valid = train_test_split(
        X_temp, y_temp, test_size=0.20, stratify=y_temp, random_state=42,
    )
    print(f"Split sizes: Train={len(X_train)} | Valid={len(X_valid)} | Test={len(X_test)}")
    print("-" * 50)

    num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
    cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

    X_train, X_valid, X_test, num_cols, cat_cols = apply_missing_value_filter(
        X_train, X_valid, X_test, threshold=missing_threshold,
    )
    X_train, X_valid, X_test, num_cols, cat_cols = apply_correlation_drop(
        X_train, y_train, X_valid, X_test, threshold=corr_threshold,
    )

    print("--- 4. Leakage-Free Transformations ---")
    median_imputers = {}
    if num_cols:
        print(f"Imputing {len(num_cols)} numeric columns with median...")
        for col in num_cols:
            median_value = X_train[col].median()
            median_imputers[col] = median_value
            X_train[col] = X_train[col].fillna(median_value)
            X_valid[col] = X_valid[col].fillna(median_value)
            X_test[col]  = X_test[col].fillna(median_value)

    encoders = {}
    cat_cardinalities = []
    for col in cat_cols:
        X_train[col] = X_train[col].fillna("Missing")
        X_valid[col] = X_valid[col].fillna("Missing")
        X_test[col]  = X_test[col].fillna("Missing")

        le = LabelEncoder()
        le.fit(X_train[col])
        new_classes = list(le.classes_)
        if "Missing" not in new_classes:
            new_classes.append("Missing")
        new_classes.append("UNKNOWN")
        le.classes_ = np.array(new_classes)
        X_train[col] = le.transform(X_train[col])
        encoders[col] = le
        cat_cardinalities.append(len(le.classes_))

        mapping = {cls: i for i, cls in enumerate(le.classes_)}
        unknown_id = mapping["UNKNOWN"]
        X_valid[col] = (X_valid[col].fillna("Missing").astype(str)
                                       .map(mapping).fillna(unknown_id).astype(int))
        X_test[col]  = (X_test[col].fillna("Missing").astype(str)
                                       .map(mapping).fillna(unknown_id).astype(int))

    print(f"Categorical features encoded: {len(cat_cols)}")

    scaler = StandardScaler()
    if num_cols:
        print(f"Scaling {len(num_cols)} numeric columns...")
        X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
        X_valid[num_cols] = scaler.transform(X_valid[num_cols])
        X_test[num_cols]  = scaler.transform(X_test[num_cols])
    print("-" * 50)

    print("--- Final Feature Summary ---")
    print(f"Numeric features      : {len(num_cols)}")
    print(f"Categorical features  : {len(cat_cols)}")
    print(f"Cat cardinalities     : {cat_cardinalities}")
    print(f"Total features        : {X_train.shape[1]}")
    print("-" * 50)

    return (
        X_train, y_train, X_valid, y_valid, X_test, y_test,
        num_cols, cat_cols, encoders, median_imputers, scaler, cat_cardinalities,
    )


def home_credit_summary(df, num_cols, cat_cols, target="TARGET"):
    print("\n" + "=" * 60)
    print("DATASET SUMMARY")
    print("=" * 60)
    print(f"Total Rows: {df.shape[0]:,}")
    print(f"Total Columns: {df.shape[1]:,}\n")
    print("Feature Breakdown")
    print(f"  Numeric Features     : {len(num_cols)}")
    print(f"  Categorical Features : {len(cat_cols)}")
    print(f"  Binary FLAG Columns  : {len([c for c in df.columns if 'FLAG' in c])}")
    print("-" * 50)
    if target in df.columns:
        target_dist = df[target].value_counts(normalize=True) * 100
        print("Target Distribution")
        print(target_dist.to_frame("Percentage %"))
        print("-" * 50)
    print("Missing Value Summary (Top 15)")
    missing = df.isnull().mean().sort_values(ascending=False)
    print((missing * 100).head(15).to_frame("Missing %"))
    print("=" * 60)


home_credit_summary(df, num_cols, cat_cols)
(
    X_train, y_train, X_valid, y_valid, X_test, y_test,
    final_num_cols, final_cat_cols, encoders, imputers, scaler, cat_cardinalities_final,
) = preprocess_data_pipeline(df, target="TARGET", corr_threshold=0.9, missing_threshold=0.9)

print("\n================ Dataset Split Summary ================\n")
print(f"Train : X = {X_train.shape},  y = {y_train.shape}")
print(f"Valid : X = {X_valid.shape},  y = {y_valid.shape}")
print(f"Test  : X = {X_test.shape},   y = {y_test.shape}")
print(f"\nNumeric features      : {len(final_num_cols)}")
print(f"Categorical features  : {len(final_cat_cols)}")
print(f"Cat cardinalities     : {cat_cardinalities_final}")

In [ ]:
# Build torch tensors for FTT-style models.
data_numpy = {"train": {}, "val": {}, "test": {}}
data_numpy["train"]["x_cat"]  = X_train[final_cat_cols]
data_numpy["val"]["x_cat"]    = X_valid[final_cat_cols]
data_numpy["test"]["x_cat"]   = X_test[final_cat_cols]
data_numpy["train"]["X_cont"] = X_train[final_num_cols]
data_numpy["val"]["X_cont"]   = X_valid[final_num_cols]
data_numpy["test"]["X_cont"]  = X_test[final_num_cols]
data_numpy["train"]["y"]      = y_train
data_numpy["val"]["y"]        = y_valid
data_numpy["test"]["y"]       = y_test

data = {}
for part in data_numpy:
    data[part] = {
        "X_cont": torch.tensor(data_numpy[part]["X_cont"].to_numpy(), dtype=torch.float32, device=device),
        "x_cat":  torch.tensor(data_numpy[part]["x_cat"].to_numpy(),  dtype=torch.long,    device=device),
        "y":      torch.tensor(data_numpy[part]["y"].to_numpy(),      dtype=torch.float32, device=device),
    }

d_numerical = len(final_num_cols)
n_train = data["train"]["y"].shape[0]
print(f"d_numerical = {d_numerical}, n_train = {n_train}")

In [ ]:
# When DEV_MODE is on, subsample data to a few thousand rows so a full training
# pass completes in a couple of minutes. No-op when DEV_MODE is False.
if DEV_MODE:
    _n_dev = 5000
    for _part in data:
        _idx = torch.randperm(data[_part]["y"].shape[0], device=device)[:_n_dev]
        for _k in data[_part]:
            data[_part][_k] = data[_part][_k][_idx]
    print({_part: {k: v.shape for k, v in data[_part].items()} for _part in data})

## 4. Model Definition

In [ ]:
class CatBoostEncoder(nn.Module):
    """
    Lightweight target-statistics encoder for categorical features
    (approximates the CatBoost-style encoder used in the paper).
    Learns a per-category embedding table, initialised to zero.
    """
    def __init__(self, cardinalities: List[int], out_dim: int):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(card + 1, out_dim, padding_idx=card)
            for card in cardinalities
        ])
        nn.init.zeros_(self.embeddings[0].weight) # start near zero
        for emb in self.embeddings:
            nn.init.normal_(emb.weight, std=0.01)

    def forward(self, x_cat: torch.Tensor) -> torch.Tensor:
        # x_cat: [B, C_cat] -> [B, C_cat, D]
        out = torch.stack([emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)], dim=1)
        return out


class NumericalEmbedding(nn.Module):
    """Linear embedding of each numerical feature into D-dimensional space."""
    def __init__(self, num_features: int, embed_dim: int):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(num_features, embed_dim))
        self.bias = nn.Parameter(torch.zeros(num_features, embed_dim))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, F] -> [B, F, D]
        return x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)


class SemiPermeableAttention(nn.Module):
    """
    Semi-Permeable Attention (SPA) — Eq. (2-4) in the paper.
    More informative features can influence less informative ones, but
    not vice versa (upper-triangular mask after MI-sort).
    Interaction-Attenuation Initialization sets Q/K/V weights near-zero.
    """
    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.dropout = nn.Dropout(dropout)

        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

        # Interaction-Attenuation Initialization
        for proj in [self.q_proj, self.k_proj, self.v_proj]:
            nn.init.normal_(proj.weight, std=1e-4)
        nn.init.zeros_(self.out_proj.weight)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        x : [B, C, D]
        mask: [C, C] upper-triangular causal mask (True = keep, False = block)
        """
        B, C, D = x.shape
        H, Hd = self.num_heads, self.head_dim

        q = self.q_proj(x).view(B, C, H, Hd).transpose(1, 2) # [B,H,C,Hd]
        k = self.k_proj(x).view(B, C, H, Hd).transpose(1, 2)
        v = self.v_proj(x).view(B, C, H, Hd).transpose(1, 2)

        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale # [B,H,C,C]

        if mask is not None:
            # mask: [C,C] bool – positions to MASK OUT set to -inf
            attn = attn.masked_fill(~mask.unsqueeze(0).unsqueeze(0), float("-inf"))

        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v) # [B,H,C,Hd]
        out = out.transpose(1, 2).reshape(B, C, D)
        return self.out_proj(out)


class AttentiveFFN(nn.Module):
    """
    Attentive Feedforward (AIUM) — boosts fitting capability (Problem P3).
    Uses a gating mechanism: output = gate(x) ⊙ transform(x).
    """
    def __init__(self, embed_dim: int, expansion: int = 4, dropout: float = 0.0):
        super().__init__()
        hidden = embed_dim * expansion
        self.fc1 = nn.Linear(embed_dim, hidden)
        self.gate = nn.Linear(embed_dim, hidden)
        self.fc2 = nn.Linear(hidden, embed_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        g = torch.sigmoid(self.gate(x))
        h = F.gelu(self.fc1(x)) * g
        return self.fc2(self.drop(h))


class ExcelFormerBlock(nn.Module):
    """One ExcelFormer Transformer block: SPA + AIUM with pre-LN."""
    def __init__(self, embed_dim: int, num_heads: int,
                 diam_dropout: float = 0.0,
                 aium_dropout: float = 0.0,
                 residual_dropout: float = 0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = SemiPermeableAttention(embed_dim, num_heads, diam_dropout)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffn = AttentiveFFN(embed_dim, dropout=aium_dropout)
        self.drop = nn.Dropout(residual_dropout)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        x = x + self.drop(self.attn(self.ln1(x), mask))
        x = x + self.drop(self.ffn(self.ln2(x)))
        return x


class ExcelFormer(nn.Module):
    """
    Full ExcelFormer model.

    Architecture:
        1. Embed numerical and categorical features -> [B, C_total, D]
        2. MI-based feature sort (approximated by learnable sort during training)
        3. Stack of ExcelFormerBlock with SPA mask
        4. Attentive decoder -> logits
    """
    def __init__(
        self,
        num_numerical: int,
        cat_cardinalities: List[int],
        embed_dim: int = 64,
        num_layers: int = 3,
        num_heads: int = 8,
        out_channels: int = 1, # single logit → BCE, matching d_out=1 style
        diam_dropout: float = 0.1,
        aium_dropout: float = 0.1,
        residual_dropout: float = 0.0,
        mixup: Optional[str] = "feat_mix", # "feat_mix" | "hidden_mix" | None
        beta: float = 0.5,
    ):
        super().__init__()
        self.num_numerical = num_numerical
        self.mixup = mixup
        self.beta = beta
        total_cols = num_numerical + len(cat_cardinalities)
        self.total_cols = total_cols

        # ── encoders ─────────────────────────────────────────────────────────
        self.num_embed = NumericalEmbedding(num_numerical, embed_dim)
        self.cat_embed = CatBoostEncoder(cat_cardinalities, embed_dim) if cat_cardinalities else None

        # Learnable MI-sort permutation (fixed after training in the paper;
        # here we use a soft permutation via temperature-scaled scores)
        self.feature_scores = nn.Parameter(torch.randn(total_cols))

        # ── transformer blocks ────────────────────────────────────────────────
        self.blocks = nn.ModuleList([
            ExcelFormerBlock(embed_dim, num_heads, diam_dropout, aium_dropout, residual_dropout)
            for _ in range(num_layers)
        ])

        # ── decoder (attentive linear as in the paper) ────────────────────────
        self.ln_out = nn.LayerNorm(embed_dim)
        self.attn_w = nn.Linear(embed_dim, 1)
        self.head = nn.Linear(embed_dim, out_channels)

        self.register_buffer("spa_mask", self._build_spa_mask(total_cols))

    # ── helpers ───────────────────────────────────────────────────────────────
    @staticmethod
    def _build_spa_mask(n: int) -> torch.Tensor:
        """Upper-triangular mask: feature i can attend to feature j only if j <= i
        (after sorting from most→least informative)."""
        return torch.ones(n, n, dtype=torch.bool).tril()

    def _feat_mix(self, x: torch.Tensor, y: torch.Tensor):
        """Feat-Mix: mix raw feature embeddings between two samples."""
        if not self.training:
            return x, y, None
        lam = np.random.beta(self.beta, self.beta)
        B = x.size(0)
        idx = torch.randperm(B, device=x.device)
        x_mix = lam * x + (1 - lam) * x[idx]
        y_mix = (y, y[idx], lam)
        return x_mix, y_mix, idx

    def _hidden_mix(self, x: torch.Tensor, y: torch.Tensor):
        """Hidden-Mix: mix hidden representations mid-forward."""
        if not self.training:
            return x, y, None
        lam = np.random.beta(self.beta, self.beta)
        B = x.size(0)
        idx = torch.randperm(B, device=x.device)
        x_mix = lam * x + (1 - lam) * x[idx]
        return x_mix, (y, y[idx], lam), idx

    def forward(self, x_num: torch.Tensor, x_cat: Optional[torch.Tensor] = None,
                y: Optional[torch.Tensor] = None):
        """
        Returns logits (and mixed y during training if mixup is active).
        """
        # ── embed ─────────────────────────────────────────────────────────────
        parts = [self.num_embed(x_num)] # [B, Cn, D]
        if self.cat_embed is not None and x_cat is not None:
            parts.append(self.cat_embed(x_cat)) # [B, Cc, D]
        x = torch.cat(parts, dim=1) # [B, C, D]

        # ── Feat-Mix augmentation ─────────────────────────────────────────────
        mixed_y = y
        if self.mixup == "feat_mix" and y is not None:
            x, mixed_y, _ = self._feat_mix(x, y)

        # ── transformer blocks ────────────────────────────────────────────────
        for i, block in enumerate(self.blocks):
            # Hidden-Mix applied after 1st block
            if self.mixup == "hidden_mix" and i == 1 and y is not None:
                x, mixed_y, _ = self._hidden_mix(x, y)
            x = block(x, self.spa_mask)

        # ── attentive decoder ─────────────────────────────────────────────────
        x = self.ln_out(x) # [B, C, D]
        w = torch.softmax(self.attn_w(x), dim=1) # [B, C, 1]
        out = (w * x).sum(dim=1) # [B, D]
        logits = self.head(out) # [B, out_channels]

        if y is not None and self.training and self.mixup is not None and isinstance(mixed_y, tuple):
            return logits, mixed_y
        return logits, None


# ─────────────────────────────────────────────────────────────────────────────
# 3. MIXUP LOSS (BCE-based, single logit)
# ─────────────────────────────────────────────────────────────────────────────

def mixup_bce(logits: Tensor, mixed_y, weights: Optional[Tensor] = None) -> Tensor:
    """
    Binary cross-entropy mixup loss.
    mixed_y is either:
      • a plain float tensor → standard weighted BCE
      • a tuple (y_a, y_b, lam) → interpolated BCE
    """
    if isinstance(mixed_y, tuple):
        y_a, y_b, lam = mixed_y
        loss_a = F.binary_cross_entropy_with_logits(logits, y_a.float(), weight=weights)
        loss_b = F.binary_cross_entropy_with_logits(logits, y_b.float(), weight=weights)
        return lam * loss_a + (1 - lam) * loss_b
    return F.binary_cross_entropy_with_logits(logits, mixed_y.float(), weight=weights)

In [ ]:

def mixup_bce(logits: Tensor, mixed_y, weights: Optional[Tensor] = None) -> Tensor:
    """
    Binary cross-entropy mixup loss.
    mixed_y is either:
      • a plain float tensor → standard weighted BCE
      • a tuple (y_a, y_b, lam) → interpolated BCE
    """
    if isinstance(mixed_y, tuple):
        y_a, y_b, lam = mixed_y
        loss_a = F.binary_cross_entropy_with_logits(logits, y_a.float(), weight=weights)
        loss_b = F.binary_cross_entropy_with_logits(logits, y_b.float(), weight=weights)
        return lam * loss_a + (1 - lam) * loss_b
    return F.binary_cross_entropy_with_logits(logits, mixed_y.float(), weight=weights)

## 5. Training Constants & Helpers

In [ ]:
# BATCH_SIZE = 1024 # 8 192 — same as reference
N_EPOCHS = 100
PATIENCE = 10
POS_WEIGHT = 6.0 # class weight for positive class
# PARQUET_PATH = "processed_data_densest.parquet.gzip" # real file or synthetic
RESULTS_JSON = str(RESULTS_DIR / "excelformer_hc_trials.json")
GLOBAL_MODEL_PATH = str(MODELS_DIR / "excelformer_hc_best.pth")
import json
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trial_results: List[dict] = []
global_best_auc = 0.774

In [ ]:
def get_dynamic_batch_size(
    d_token: int,
    n_blocks: int,
    d_numerical: int,
    cat_cardinalities: list,
    target_gpu_gb: float = 15.0,
    safety_factor: float = 0.55, # more conservative default
) -> int:
    total_cols = d_numerical + len(cat_cardinalities)
    B = 4 # bytes per float32

    # ── model weights (static, always resident) ───────────────────────────────
    # embeddings
    num_embed_params = d_numerical * d_token * 2 # weight + bias
    cat_embed_params = sum(cat_cardinalities) * d_token
    # per block: SPA (4 projections) + FFN (fc1, gate, fc2) + 2x LayerNorm
    spa_params = 4 * (d_token * d_token)
    ffn_params = 2 * (d_token * d_token * 4) + (d_token * 4 * d_token)
    ln_params = 2 * d_token * 2
    block_params = (spa_params + ffn_params + ln_params) * n_blocks
    # decoder
    decoder_params = d_token + d_token * 1
    total_params = num_embed_params + cat_embed_params + block_params + decoder_params
    model_weights_mem = total_params * B

    # ── optimizer states (Adam: m + v per param, same size as weights) ────────
    optimizer_mem = model_weights_mem * 2

    # ── activations per sample (forward pass) ─────────────────────────────────
    embed_act = total_cols * d_token * B
    # attention: Q, K, V tensors + attn map [total_cols, total_cols]
    attn_act = (3 * total_cols * d_token + total_cols * total_cols) * B
    ffn_act = total_cols * d_token * 4 * B * 2 # fc1 + gate hidden
    block_act = (embed_act + attn_act + ffn_act) * n_blocks
    decoder_act = total_cols * d_token * B
    forward_per_sample = embed_act + block_act + decoder_act

    # ── gradients ≈ same as forward activations ────────────────────────────────
    backward_per_sample = forward_per_sample

    total_per_sample = forward_per_sample + backward_per_sample

    # ── CUDA overhead + torch.compile buffers (empirically ~1.5–2 GB) ─────────
    cuda_overhead_bytes = 2.0 * (1024 ** 3)

    budget_bytes = (target_gpu_gb * (1024 ** 3) * safety_factor) - cuda_overhead_bytes - model_weights_mem - optimizer_mem

    if budget_bytes <= 0:
        return 64 # fallback minimum

    raw = int(budget_bytes / total_per_sample)
    batch_size = max(64, 1 << (raw.bit_length() - 1)) # round down to power of 2

    print(f"[BatchSizer] total_cols={total_cols}, d_token={d_token}, n_blocks={n_blocks}")
    print(f" model={model_weights_mem/1e6:.1f}MB optim={optimizer_mem/1e6:.1f}MB "
          f"cuda_overhead=2000MB")
    print(f" per_sample={total_per_sample/1e3:.2f}KB "
          f"budget={budget_bytes/1e9:.2f}GB raw={raw} BATCH_SIZE={batch_size}")

    return batch_size

In [ ]:
n_train = data["train"]["y"].shape[0]

## 6. Optuna Optimization

In [ ]:
def optim(trial: optuna.Trial) -> float:
    global global_best_auc

    # ── search space ──────────────────────────────────────────────────────────
    d_token = trial.suggest_categorical("d_token", [64, 128])
    n_blocks = trial.suggest_int( "n_blocks", 2, 4)
    attention_heads = trial.suggest_categorical("attn_heads", [2, 4, 8])
    diam_dropout = trial.suggest_float( "diam_drop", 0.0, 0.1)
    aium_dropout = trial.suggest_float( "aium_drop", 0.0, 0.1)
    residual_dropout = trial.suggest_float( "res_drop", 0.0, 0.1)
    mixup_type = trial.suggest_categorical("mixup", ["feat_mix", "hidden_mix", "none"])
    beta = trial.suggest_categorical("beta", [0.5, 0.8, 1.0])
    lr = trial.suggest_float( "lr", 1e-4, 5e-3, log=True)

    if mixup_type == "none":
        mixup_type = None
    BATCH_SIZE = get_dynamic_batch_size(
        d_token = d_token,
        n_blocks = n_blocks,
        d_numerical = d_numerical,
        cat_cardinalities = cat_cardinalities_final,
        target_gpu_gb = 15.0,
        safety_factor = 0.6, # leave 40% headroom for misc allocations
    )

    warmup_end = int(N_EPOCHS * 0.2) + 5

    # ── print trial header (mirrors reference style) ──────────────────────────
    print("\n" + "=" * 70)
    print(f" Starting Trial {trial.number}")
    print(" Hyperparameters:")
    print(f" d_token = {d_token}")
    print(f" n_blocks = {n_blocks}")
    print(f" attention_heads = {attention_heads}")
    print(f" diam_dropout = {diam_dropout:.3f}")
    print(f" aium_dropout = {aium_dropout:.3f}")
    print(f" residual_dropout = {residual_dropout:.3f}")
    print(f" beta = {beta:.2f}")
    print(f" mixup = {mixup_type}")
    print(f" lr = {lr:.2e}")
    print("=" * 70 + "\n")

    # ── model ─────────────────────────────────────────────────────────────────
    model = ExcelFormer(
        num_numerical = d_numerical,
        cat_cardinalities= cat_cardinalities_final,
        embed_dim = d_token,
        num_layers = n_blocks,
        num_heads = attention_heads,
        out_channels = 1, # single logit → BCE
        diam_dropout = diam_dropout,
        aium_dropout = aium_dropout,
        residual_dropout = residual_dropout,
        mixup = mixup_type,
        beta = beta, #0.5,
    ).to(device)
    # model = torch.compile(model, mode="default")

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parameters: {n_params:,}\n")

    # ── optimizer + OneCycleLR (per-batch step, mirrors reference) ────────────
    epoch_size = math.ceil(n_train / BATCH_SIZE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr = lr * 5, # 5× base, same ratio as reference
        steps_per_epoch = epoch_size,
        epochs = N_EPOCHS,
        pct_start = 0.2,
        anneal_strategy = "cos",
    )

    loss_fn = F.binary_cross_entropy_with_logits

    # ── apply_model closure (mirrors reference exactly) ───────────────────────
    def apply_model(batch: Dict[str, Tensor]) -> Tensor:
        logits, _ = model(batch["X_cont"], batch.get("x_cat"))
        return logits.squeeze(-1) # [B] — single logit per sample

    # ── evaluate closure (mirrors reference exactly) ──────────────────────────
    @torch.no_grad()
    def evaluate(part: str) -> float:
        model.eval()
        y_pred = (
            torch.cat(
                [apply_model(batch)
                 for batch in delu.iter_batches(data[part], BATCH_SIZE)]
            )
            .cpu()
            .numpy()
        )
        y_true = data[part]["y"].cpu().numpy()
        y_pred = scipy.special.expit(y_pred) # sigmoid → probability
        return roc_auc_score(y_true, y_pred)

    # ── pre-training baseline ─────────────────────────────────────────────────
    print(f"Test score before training: AUC Val = {evaluate('val'):.4f}")
    print(f"Device: {device.type.upper()}")
    print("-" * 88 + "\n")

    # ── delu early stopping + timer ───────────────────────────────────────────
    timer = delu.tools.Timer()
    early_stopping = delu.tools.EarlyStopping(PATIENCE, mode="max")
    best = {"val": -math.inf, "test": -math.inf, "epoch": -1}
    trial_pruned=False
    timer.run()

    # ── training loop ─────────────────────────────────────────────────────────
    for epoch in range(N_EPOCHS):
      in_warmup = epoch < warmup_end
      for batch in tqdm(
          delu.iter_batches(data["train"], BATCH_SIZE, shuffle=True),
          desc=f"Epoch {epoch}",
          total=epoch_size,
      ):
          model.train()
          optimizer.zero_grad()

          y_batch = batch["y"].float()

          # Class weights — mirrors reference's weighted BCE
          weights = torch.where(
              y_batch == 1,
              torch.tensor(POS_WEIGHT, device=y_batch.device),
              torch.tensor(1.0, device=y_batch.device),
          )

          # raw_logits = apply_model(batch)

          # Mixup augmentation (if active the model returns mixed_y)
          logits, mixed_y = model(batch["X_cont"], batch.get("x_cat"), y_batch)
          logits = logits.squeeze(-1)

          if mixed_y is not None:
              loss = mixup_bce(logits, mixed_y, weights=weights) # weights inside mixup
          else:
              loss = loss_fn(logits, y_batch, weight=weights)

          loss.backward()
          nn.utils.clip_grad_norm_(model.parameters(), 1.0)
          optimizer.step()
          scheduler.step() # ← per-batch, same as reference

      # ── epoch-end metrics ─────────────────────────────────────────────────
      val_auc = evaluate("val")
      test_auc = evaluate("test")
      print(f" AUC Val {val_auc:.4f} Test {test_auc:.4f} [time] {timer}")

      if val_auc > best["val"]:
          print(" New best epoch! ")
          best = {"val": val_auc, "test": test_auc, "epoch": epoch}

          if val_auc > global_best_auc:
              torch.save(model.state_dict(), GLOBAL_MODEL_PATH+'_trial_'+str(trial.number)+'.pth')
              global_best_auc = val_auc
              print(f" Saved to {GLOBAL_MODEL_PATH+'_trial_'+str(trial.number)} (global best {global_best_auc:.4f})")

      trial.report(val_auc, step=epoch)
      if not in_warmup and trial.should_prune():
          print(f" ️ Trial {trial.number} pruned at epoch {epoch}")
          trial_pruned=True
          break

      # ── early stopping ────────────────────────────────────────────────────
      early_stopping.update(val_auc)
      if not in_warmup:
        if early_stopping.should_stop():
            print("⏹ Early stopping triggered.")
            break

    # ── persist trial result ──────────────────────────────────────────────────
    trial_results.append({
        "trial_number": trial.number,
        "trial_type": "ExcelFormer SPA + AIUM",
        "auc_val": val_auc,
        "auc_test": test_auc,
        "d_token": d_token,
        "n_blocks": n_blocks,
        "attn_heads": attention_heads,
        "diam_dropout": diam_dropout,
        "aium_dropout": aium_dropout,
        "residual_dropout":residual_dropout,
        "mixup": str(mixup_type),
        "lr": lr,
        "best": best,
        "time": str(timer),
        'beta': beta,
        'trial_pruned': 'yes' if trial_pruned else 'no',
        'stopping_epoch': epoch
    })
    with open(RESULTS_JSON+'_trial_'+str(trial.number)+'.json', "w") as f:
        json.dump(trial_results, f, indent=4)
    files.download(RESULTS_JSON+'_trial_'+str(trial.number)+'.json')

    if trial_pruned:
      raise optuna.exceptions.TrialPruned()

    print("\n\nResult:")
    print(best)
    return best["val"]
pruner = optuna.pruners.MedianPruner(
    n_startup_trials=3,
    n_warmup_steps=int(N_EPOCHS * 0.2) + 5,
    interval_steps=3,
)
storage_name = "sqlite:///excelformer_study.db"
study = optuna.create_study(direction="maximize",
                            study_name="excelformer_home",
                            storage=storage_name,
                            load_if_exists=True,
                            pruner=pruner)
print(f"Study resumed. Number of finished trials: {len(study.trials)}")
try:
    global_best_auc = study.best_value
    print(f"Resuming with global best AUC: {global_best_auc:.4f}")
except ValueError:
    # If the study was brand new and has no trials yet
    global_best_auc = -float('inf')
study.optimize(optim, n_trials=(1 if DEV_MODE else 30))

## 7. Persistent Study Setup

In [ ]:
storage_name = "sqlite:///excelformer_study.db"
persistent_study = optuna.create_study(
    study_name="excelformer_home",
    storage=storage_name,
    direction="maximize",
    load_if_exists=True
)
for trial in study.trials:
    try:
        persistent_study.add_trial(trial)
    except ValueError:
        # This handles cases where a trial with the same number might already exist
        print(f"Trial {trial.number} already exists in storage. Skipping.")